# ECG Arrhythmia Classification — MIT-BIH (Kaggle)### Inter-patient vs intra-patient evaluation, multi-scale 1-D CNN**Before running anything, open the right-hand panel and set:**| Setting | Value | Why ||---|---|---|| **Accelerator** | GPU T4 x2 | ~30 min instead of ~3 hours || **Internet** | **On** | Needed to `pip install wfdb` and download MIT-BIH from PhysioNet || **Persistence** | Files only (optional) | Keeps `/kaggle/working` between sessions |Internet is **off** by default on Kaggle and is the single most common reason this fails.You may need to verify your phone number on your Kaggle account to enable it.

## 1 · Check the environment

In [ ]:
import sys, os, subprocess, globprint("Python", sys.version.split()[0])import torchprint("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())if torch.cuda.is_available():    print("GPU:", torch.cuda.get_device_name(0))else:    print("\n!! No GPU. Right panel -> Accelerator -> GPU T4 x2, then re-run.")# Internet check — this is the usual failure point on Kaggle.try:    import urllib.request    urllib.request.urlopen("https://physionet.org", timeout=15)    print("Internet: ON — PhysioNet reachable")except Exception as e:    print("\n!! Internet appears OFF:", e)    print("   Right panel -> Internet -> On. (Account phone verification may be required.)")

## 2 · Install wfdb

In [ ]:
!pip install -q wfdbimport wfdb; print("wfdb", wfdb.__version__)

## 3 · Locate the projectUpload `ecg-arrhythmia-dl.zip` first: **+ Add Input → Upload → Dataset**, drag the zip in,give it a title, click Create. It appears under `/kaggle/input/<your-title>/`.This cell finds it wherever it landed — zipped or already extracted — and copies it into`/kaggle/working` so the scripts can write results.

In [ ]:
import os, glob, zipfile, shutilWORK = "/kaggle/working"os.chdir(WORK)def find_root(base):    """Return the folder containing src/config.py, or None."""    for d, _, files in os.walk(base):        if os.path.basename(d) == "src" and "config.py" in files:            return os.path.dirname(d)    return Noneroot = find_root(WORK)# Not in working yet — pull it out of /kaggle/input.if root is None:    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)    print("zip files in /kaggle/input:", zips)    if zips:        with zipfile.ZipFile(zips[0]) as z:            z.extractall(WORK)        root = find_root(WORK)# Uploaded as a plain folder rather than a zip — copy it in (input is read-only).if root is None:    src_root = find_root("/kaggle/input")    if src_root:        dst = os.path.join(WORK, os.path.basename(src_root))        shutil.copytree(src_root, dst, dirs_exist_ok=True)        root = dstif root is None:    print("\n!! Project not found.")    print("   Contents of /kaggle/input:")    for p in glob.glob("/kaggle/input/*"):        print("    ", p)    raise SystemExit("Upload ecg-arrhythmia-dl.zip via + Add Input -> Upload -> Dataset")os.chdir(root)print("\nProject root:", os.getcwd())print(sorted(os.listdir()))

## 4 · Sanity check (≈ 1 minute, synthetic data — no download)Catches a broken environment before the long run.

In [ ]:
!python tests/test_pipeline.py

In [ ]:
!python scripts/run_experiment.py --smoke-test --epochs 3

## 5 · Download MIT-BIH and segment beats (≈ 5 min)

In [ ]:
!python scripts/prepare_data.py

## 6 · Run the experiment grid9 configurations × 3 seeds. Roughly 30 min on a T4.Faster first pass: `!python scripts/run_ablation.py --seeds 1`

In [ ]:
!python scripts/run_ablation.py

## 7 · Build the report

In [ ]:
!python scripts/make_report.py

## 8 · Confirm every requirement is metShould print **0 waiting · 0 failed · READY TO SUBMIT**.

In [ ]:
!python scripts/verify_submission.py

## 9 · Look at the results

In [ ]:
from IPython.display import Markdown, displaydisplay(Markdown(open("RESULTS.md").read()))

In [ ]:
from pathlib import Pathfrom IPython.display import Image, displayfor p in sorted(Path("results/figures").glob("fig*.png")):    print("\n" + p.name)    display(Image(str(p)))

## 10 · Get the files outKaggle exposes `/kaggle/working` in the **Output** tab on the right — but only files at thetop level are easy to grab, so this bundles everything into one zip there.

In [ ]:
import shutil, osshutil.make_archive("/kaggle/working/ecg_results", "zip", os.getcwd(), "results")shutil.copy("RESULTS.md", "/kaggle/working/RESULTS.md")shutil.copy("README.md", "/kaggle/working/README.md")print("Ready in the Output tab (right panel):")for f in ["ecg_results.zip", "RESULTS.md", "README.md"]:    p = "/kaggle/working/" + f    print(f"  {f:<20} {os.path.getsize(p):>10,} bytes")print("\nClick the download arrow next to each, or Save Version -> Output to keep them.")

---### Troubleshooting| Symptom | Fix ||---|---|| `SyntaxError` on a comment line | You deleted a `#` from an instruction line — put it back || `No module named google.colab` | That cell is Colab-only; this notebook doesn't use it || PhysioNet download fails / hangs | Internet is off. Right panel → Internet → On (needs phone verification) || `No module named src` | Re-run cell 3; `os.chdir` did not land in the project root || `Project not found` | Upload the zip: + Add Input → Upload → Dataset || Very slow | No GPU. Right panel → Accelerator → GPU T4 x2 || CUDA out of memory | Restart the session, then `!python scripts/run_ablation.py --seeds 1` || Session died mid-run | Kaggle caps GPU hours weekly. Re-run with `--seeds 1 --epochs 20` |